## import

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import os
import sys
sys.path.append(f'{os.getcwd()}/irc_gym')
from ult import *
from irc_gym.irc.model import FuncBeliefModel
from stable_baselines3 import PPO
from auditoryforage.AF_env import AuditoryForagingReward2 as AFR2

# peroid

In [27]:
modelname = 'varynode'
iepoch=51
# training hyper params
epoch_size = 33333
n_epoch = 55
n_seed = 1
seed = 0
no_episodes = 3333  # for eval plot 
plot_no_episodes = 555
vmin, vmax = 10, 10000
food_reward_list = np.linspace(vmin, vmax, 5)
attcost = -1.6
facost = -50
p1p2 = np.array([.5, .7])
env_param = [0, None, attcost, .25, facost, 0, 0]
np.random.seed(seed)
node_list = [10, 25, 40]


In [28]:
results=[]
for inode, node in enumerate(node_list):
    task = AFR2(spec={'agent': 
                                        {'lick_cost': env_param[0],
                                        'food_reward': env_param[1],
                                        'attention_cost_coeff': env_param[2],
                                        'attention_cost_temp': env_param[3],
                                        'penalty_cost': env_param[4],
                                        'iti_cost': env_param[5],
                                        'time_in_game_reward': env_param[6]},
                                'experiment':{                              'no_signal_nodes':node}})
    task.food_reward_list = food_reward_list
    taskbelief = FuncBeliefModel(env=task, rng=1)
    task.obs_certainity_possible = p1p2
    thismodel = f'seed_0_{modelname}_{node}'
    model = PPO.load(f'ycstore/{thismodel}_epoch_{iepoch}')

    def eval_wrapper(a):
        episode = run_one_episode(task=task, taskbelief=taskbelief, agent=model,
                                num_steps=100000, deterministic=True)
        return episode

    with multiprocess.Pool(processes=8) as pool:
        all_episode_data = pool.map(eval_wrapper, range(no_episodes))
    
    for episode in all_episode_data:
        # process
        episode['inode'] = inode
        episode['at_times'] = np.where(
            np.array([elt[1] for elt in episode['actions']]) == 1)[0].astype('int')
        episode['at_seq'] = [float(elt[1])
                             for elt in episode['actions']]
        if episode['actions'][-1][0] == 1:
            episode['licktime'] = int(len(episode['states']))
        else:
            episode['lick_licktimetime'] = -1
        episode['trial_len'] = len(episode['actions'])

        results.append(episode)

df = pd.DataFrame(results)

KeyboardInterrupt: 

In [ ]:

vmin,vmax=0,50
nbin=20
for food_reward_idx in range(len(food_reward_list)):
    for inode, node in enumerate(node_list):
        grid=np.zeros((vmax, nbin))
        bins=np.linspace(0,1,nbin)
        bindata=[[] for _ in range(nbin)] 
        subdf=df[(df.trial_food_reward_idx==food_reward_idx)&(df.inode==inode)]
        sb, nextatt=[],[]
        for alist, b in zip(subdf.at_times, subdf.beliefs):
            try:
                a,b=trialnextatt(alist,b),trialsb(b)  
                for j in range(len(b)):
                    thisa,thisb=a[j], b[j]
                    for i, (s,e) in enumerate(zip(bins, bins[1:])):
                        if s<thisb<=e:
                            bindata[i].append(thisa)
            except:
                continue
        for i in range(nbin):
            count=Counter(bindata[i])
            col=[]
            for j in range(vmin, vmax):
                col.append(count[j])
            col=np.array(col)
            col=col/np.sum(col)
            grid[:,i]=col
        fig=plt.figure()
        # grid=np.log(grid)
        c=plt.imshow(grid,aspect='auto', origin='lower')
        plt.colorbar(c)
        plt.xlabel('prob')
        plt.ylabel('gap')
        plt.title(f'reward id: {food_reward_idx}, \np:{p1p2},\nnode:{node}')
        plt.xticks([0,nbin],[0,1])
        quicksave(f'vary node, reward id {food_reward_idx}, node {node}', 'test', fig=fig)
        # plt.show()
        plt.close()

vmin,vmax=0,50
nbin=20
for food_reward_idx in range(len(food_reward_list)):
    for inode, node in enumerate(node_list):
        grid=np.zeros((vmax, nbin))
        bins=np.linspace(0,1,nbin)
        bindata=[[] for _ in range(nbin)] 
        subdf=df[(df.trial_food_reward_idx==food_reward_idx)&(df.inode==inode)]
        sb, nextatt=[],[]
        for alist, b in zip(subdf.at_times, subdf.beliefs):
            try:
                a,b=trialnextatt(alist,b),trialsb(b)  
                for j in range(len(b)):
                    thisa,thisb=a[j], b[j]
                    for i, (s,e) in enumerate(zip(bins, bins[1:])):
                        if s<thisb<=e:
                            bindata[i].append(thisa)
            except:
                continue
        for i in range(nbin):
            count=Counter(bindata[i])
            col=[]
            for j in range(vmin, vmax):
                col.append(count[j])
            col=np.array(col)
            col=col/np.sum(col)
            grid[:,i]=col
        fig=plt.figure()
        grid=np.log(grid)
        c=plt.imshow(grid,aspect='auto', origin='lower')
        plt.colorbar(c)
        plt.xlabel('prob')
        plt.ylabel('gap')
        plt.title(f'reward id: {food_reward_idx}, \np:{p1p2},\nnode:{node}')
        plt.xticks([0,nbin],[0,1])
        quicksave(f'vary node, reward id {food_reward_idx}, node {node} log', 'test', fig=fig)
        # plt.show()
        plt.close()


/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/1624497308.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/1624497308.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/1624497308.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/1624497308.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/1624497308.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT su

# np

In [36]:
modelname = 'vary node np v2'
iepoch=15
# training hyper params
epoch_size = 33333
n_epoch = 55
n_seed = 1
seed = 0
no_episodes = 3333  # for eval plot 
plot_no_episodes = 555
vmin, vmax = 10, 10000
food_reward_list = np.linspace(vmin, vmax, 5)
attcost = -1.6
facost = -50
p1p2 = np.array([.6, .7])
env_param = [0, None, attcost, .25, facost, 0, 0]
np.random.seed(seed)
node_list = [10, 25, 40]


In [37]:
results=[]
for inode, node in enumerate(node_list):
    task = AFR2(spec={'agent': 
                                        {'lick_cost': env_param[0],
                                        'food_reward': env_param[1],
                                        'attention_cost_coeff': env_param[2],
                                        'attention_cost_temp': env_param[3],
                                        'penalty_cost': env_param[4],
                                        'iti_cost': env_param[5],
                                        'time_in_game_reward': env_param[6]},
                                'experiment':{                              'no_signal_nodes':node}})
    task.food_reward_list = food_reward_list
    taskbelief = FuncBeliefModel(env=task, rng=1)
    task.obs_certainity_possible = p1p2
    thismodel = f'seed{seed}_{modelname}_n{node}_ep{iepoch}'
    model = PPO.load(f'ycstore/{thismodel}')

    def eval_wrapper(a):
        episode = run_one_episode(task=task, taskbelief=taskbelief, agent=model,
                                num_steps=100000, deterministic=True)
        return episode

    with multiprocess.Pool(processes=8) as pool:
        all_episode_data = pool.map(eval_wrapper, range(no_episodes))
    
    for episode in all_episode_data:
        # process
        episode['inode'] = inode
        episode['at_times'] = np.where(
            np.array([elt[1] for elt in episode['actions']]) == 1)[0].astype('int')
        episode['at_seq'] = [float(elt[1])
                             for elt in episode['actions']]
        if episode['actions'][-1][0] == 1:
            episode['licktime'] = int(len(episode['states']))
        else:
            episode['lick_licktimetime'] = -1
        episode['trial_len'] = len(episode['actions'])

        results.append(episode)

df = pd.DataFrame(results)

In [38]:

vmin,vmax=0,50
nbin=20
for food_reward_idx in range(len(food_reward_list)):
    for inode, node in enumerate(node_list):
        grid=np.zeros((vmax, nbin))
        bins=np.linspace(0,1,nbin)
        bindata=[[] for _ in range(nbin)] 
        subdf=df[(df.trial_food_reward_idx==food_reward_idx)&(df.inode==inode)]
        sb, nextatt=[],[]
        for alist, b in zip(subdf.at_times, subdf.beliefs):
            try:
                a,b=trialnextatt(alist,b),trialsb(b)  
                for j in range(len(b)):
                    thisa,thisb=a[j], b[j]
                    for i, (s,e) in enumerate(zip(bins, bins[1:])):
                        if s<thisb<=e:
                            bindata[i].append(thisa)
            except:
                continue
        for i in range(nbin):
            count=Counter(bindata[i])
            col=[]
            for j in range(vmin, vmax):
                col.append(count[j])
            col=np.array(col)
            col=col/np.sum(col)
            grid[:,i]=col
        fig=plt.figure()
        # grid=np.log(grid)
        c=plt.imshow(grid,aspect='auto', origin='lower')
        plt.colorbar(c)
        plt.xlabel('prob')
        plt.ylabel('gap')
        plt.title(f'reward id: {food_reward_idx}, \np:{p1p2},\nnode:{node}')
        plt.xticks([0,nbin],[0,1])
        quicksave(f'vary node np, reward id {food_reward_idx}, node {node}', 'test', fig=fig)
        # plt.show()
        plt.close()

vmin,vmax=0,50
nbin=20
for food_reward_idx in range(len(food_reward_list)):
    for inode, node in enumerate(node_list):
        grid=np.zeros((vmax, nbin))
        bins=np.linspace(0,1,nbin)
        bindata=[[] for _ in range(nbin)] 
        subdf=df[(df.trial_food_reward_idx==food_reward_idx)&(df.inode==inode)]
        sb, nextatt=[],[]
        for alist, b in zip(subdf.at_times, subdf.beliefs):
            try:
                a,b=trialnextatt(alist,b),trialsb(b)  
                for j in range(len(b)):
                    thisa,thisb=a[j], b[j]
                    for i, (s,e) in enumerate(zip(bins, bins[1:])):
                        if s<thisb<=e:
                            bindata[i].append(thisa)
            except:
                continue
        for i in range(nbin):
            count=Counter(bindata[i])
            col=[]
            for j in range(vmin, vmax):
                col.append(count[j])
            col=np.array(col)
            col=col/np.sum(col)
            grid[:,i]=col
        fig=plt.figure()
        grid=np.log(grid)
        c=plt.imshow(grid,aspect='auto', origin='lower')
        plt.colorbar(c)
        plt.xlabel('prob')
        plt.ylabel('gap')
        plt.title(f'reward id: {food_reward_idx}, \np:{p1p2},\nnode:{node}')
        plt.xticks([0,nbin],[0,1])
        quicksave(f'vary node np, reward id {food_reward_idx}, node {node} log', 'test', fig=fig)
        # plt.show()
        plt.close()


/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/3816081360.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/3816081360.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/3816081360.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/3816081360.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT subset; don't know how to subset; dropped
/var/folders/93/7tm1cj7d04l46ys7ndh7qnz80000gn/T/ipykernel_67877/3816081360.py:26: RuntimeWarning: invalid value encountered in divide
  col=col/np.sum(col)
webf NOT su